Step 1: Environment Setup & Gemini API Initialization


In [ ]:
# Install required libraries
!pip install -q google-generativeai langchain langchain-google-genai chromadb pydantic langchain_community
# Remove conflicting versions
!pip uninstall -y langchain langchain-core langchain-community langchain-google-genai chromadb

# Install compatible versions
!pip install -q \
langchain==0.3.25 \
langchain-core==0.3.59 \
langchain-community==0.3.24 \
langchain-google-genai==2.1.4 \
chromadb==0.5.23

import os
from google.colab import userdata
import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI

# Students must save their API key in Colab's "Secrets" tab as 'GEMINI_API_KEY'
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

# Initialize the Gemini LLM via Langchain
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# Test the connection
response = llm.invoke("What is the capital of Sierra Leone?")
print(response.content)

Found existing installation: langchain 1.2.15
Uninstalling langchain-1.2.15:
  Successfully uninstalled langchain-1.2.15
Found existing installation: langchain-core 1.4.0
Uninstalling langchain-core-1.4.0:
  Successfully uninstalled langchain-core-1.4.0
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: langchain-google-genai 4.2.3
Uninstalling langchain-google-genai-4.2.3:
  Successfully uninstalled langchain-google-genai-4.2.3
Found existing installation: chromadb 1.5.9
Uninstalling chromadb-1.5.9:
  Successfully uninstalled chromadb-1.5.9
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.7/437.7 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


The capital of Sierra Leone is **Freetown**.


# Step 2: Prompt Templates & Chains

In [ ]:
from langchain_core.prompts import PromptTemplate

# Create a prompt template for auditing data
audit_prompt = PromptTemplate.from_template(
    "You are an IT Audit Supervisor. Analyze the following system log and identify any anomalies: {log_data}"
)

# Create a chain
audit_chain = audit_prompt | llm

# Run the chain
sample_log = "User Admin logged in at 02:00 AM from IP 192.168.1.50. 500GB of database records were exported."
result = audit_chain.invoke({"log_data": sample_log})
print(result.content)

As an IT Audit Supervisor, this system log entry immediately raises several critical red flags. This is not merely an anomaly; it strongly suggests a potential high-severity security incident.

Here's my analysis:

---

**IT Audit Anomaly Report**

**Date:** [Current Date]
**Prepared For:** CISO, Head of IT Operations, Legal Counsel
**Prepared By:** [Your Name/IT Audit Department]
**Subject:** Urgent Analysis of System Log Anomaly - Potential Data Exfiltration

**Executive Summary:**
A critical system log entry indicates highly suspicious activity involving a privileged 'Admin' account, occurring outside of standard business hours, resulting in the export of an extremely large volume of database records (500GB). This event bears the hallmarks of a potential data breach, unauthorized data exfiltration, or malicious insider activity. Immediate investigation and containment are required.

**Detailed Analysis of Anomalies:**

1.  **User Account (Admin):**
    *   **Anomaly:** The activity 

# Step 3: Complete RAG Solution with Vector DB

In [ ]:
import os
from google.colab import userdata

# LangChain imports
from langchain_community.vectorstores import Chroma
from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# API Key
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

# LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# Dummy transport policy documents
docs = [
    Document(page_content="Route CEN-CAL begins operations at 06:00 AM and ends at 22:00 PM."),
    Document(page_content="All public transport drivers must undergo compliance training every 6 months."),
    Document(page_content="Daily revenue reports from ticketing systems must be submitted to the internal audit team by 17:00 PM.")
]

# Embeddings + Vector DB
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings
)

retriever = vectorstore.as_retriever()

# Prompt
prompt = PromptTemplate.from_template(
    """
    Answer only from the context below.

    Context:
    {context}

    Question:
    {input}
    """
)

# RAG chain
combine_docs_chain = create_stuff_documents_chain(
    llm,
    prompt
)

rag_chain = create_retrieval_chain(
    retriever,
    combine_docs_chain
)

# Query
response = rag_chain.invoke({
    "input": "When are the daily revenue reports due for the audit team?"
})

print(response["answer"])


In [ ]:
import google.generativeai as genai

print("Available Embedding Models:")
for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print(f"- {m.name}")

Available Embedding Models:
- models/gemini-embedding-001
- models/gemini-embedding-2-preview
- models/gemini-embedding-2


# Step 4: Tool Calling & ReAct Agent

In [ ]:
from langchain.agents import tool, AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

# Define schema for route query
class RouteQuery(BaseModel):
    route_name: str = Field(description="The name of the transport route")

# Existing tool for real-time status
@tool(args_schema=RouteQuery)
def check_route_status(route_name: str) -> str:
    """Checks the real-time operational status of a specific transport route."""
    if route_name == "Route CEN-CAL":
        return "Route CEN-CAL is experiencing a 15-minute delay due to traffic."
    return f"{route_name} is operating on time."

# New tool for operational hours (reflects Document 1)
@tool(args_schema=RouteQuery)
def get_route_operational_hours(route_name: str) -> str:
    """Provides the fixed operational hours for a specific transport route."""
    if route_name == "Route CEN-CAL":
        return "Route CEN-CAL begins operations at 06:00 AM and ends at 22:00 PM."
    return f"Operational hours for {route_name} are not available."

# New tool for driver training requirements (reflects Document 2)
@tool
def get_driver_training_requirements() -> str:
    """Provides information about compliance training requirements for public transport drivers."""
    return "All public transport drivers must undergo compliance training every 6 months."

# New tool for revenue report submission details (reflects Document 3)
@tool
def get_revenue_report_submission_details() -> str:
    """Provides details about the submission deadline for daily revenue reports from ticketing systems."""
    return "Daily revenue reports from ticketing systems must be submitted to the internal audit team by 17:00 PM."

tools = [
    check_route_status,
    get_route_operational_hours,
    get_driver_training_requirements,
    get_revenue_report_submission_details
]

# Setup the ReAct Agent Prompt
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful transit operations assistant. Use the provided tools to answer user queries."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# Create and execute the agent
agent = create_tool_calling_agent(llm, tools, agent_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Test the agent with various queries to demonstrate the new tools
print("\n--- Testing check_route_status ---")
agent_executor.invoke({"input": "What is the current status of Route CEN-CAL"})

print("\n--- Testing get_route_operational_hours ---")
agent_executor.invoke({"input": "What are the operational hours for Route CEN-CAL?"})

print("\n--- Testing get_driver_training_requirements ---")
agent_executor.invoke({"input": "What are the driver training requirements?"})

print("\n--- Testing get_revenue_report_submission_details ---")
agent_executor.invoke({"input": "When are the daily revenue reports due?"})

# Test with a route that doesn't exist in the dummy data for check_route_status
print("\n--- Testing unknown route for check_route_status ---")
agent_executor.invoke({"input": "What is the current status of Route B?"})

# Test with a route that doesn't exist in the dummy data for get_route_operational_hours
print("\n--- Testing unknown route for get_route_operational_hours ---")
agent_executor.invoke({"input": "What are the operational hours for Route X?"})